# 04 · Community fine-tuned checkpoints (§5.5–5.6, Appendix C): Table 4

This notebook recomputes the fine-tune campaign, in which the ROME covariance $C$ is recomputed on every checkpoint.

| Paper item | What is computed |
|---|---|
| §5.5 *Population*, Appendix C | 790 selected; 43 excluded after a failed preliminary edit; 3 unavailable; 744 evaluated (550 full, 194 adapters); 2–85 per base model |
| §5.5 *Protocol* | one fact per checkpoint from a fixed pool of 100 CounterFact facts |
| Table 4 | $n$, ROME successes, TP and FP per base model, with totals |
| §5.5 *Outcomes*, Abstract, Conclusion | 733 ROME successes (98.5%), TP 584 (78.5%), FP 159 (21.4%), split by base model |
| §5.6 | 11 failed fine-tune edits, 8 of them localized |

**Definitions.** For every checkpoint the detector runs on the unedited weights and again after one ROME edit at the $\ell^*$ of its base model.
TP: the edited checkpoint is flagged, i.e. the detector returns $\ell^*$. FP: the unedited checkpoint is flagged, i.e. the detector already returns $\ell^*$.

**Inputs:** `data/basecov-profile-checkpoints.csv` (the 790 selected checkpoints and their preliminary edit),
`data/owncov-checkpoint-results.csv` (the 744 evaluated checkpoints), `data/owncov-family-results.csv` (per base model),
and, for the comparison with the base models, the files of notebook 02.

In [ ]:
import pandas as pd
from paper_common import (MODELS, EXCLUDED, OUT, read_csv, base_model_results, inputs_used,
                          Checks, latex_rows_equal, diff_rows)

pd.set_option("display.precision", 2)
checks = Checks()

sel = read_csv("data/basecov-profile-checkpoints.csv")      # 790 selected checkpoints
ft = read_csv("data/owncov-checkpoint-results.csv")         # 744 evaluated checkpoints
fam = read_csv("data/owncov-family-results.csv").set_index("family")
for d in (sel, ft):
    assert not d.family.isin(EXCLUDED).any()
ft[["family", "model_id", "checkpoint_type", "target_layer", "baseline_selected_layer", "edited_selected_layer",
    "rome_success", "baseline_fp", "edited_gram_exact"]].head()

## Population (§5.5, Appendix C)

In [ ]:
checks.value("selected checkpoints", "§5.5, App. C", len(sel), 790)
checks.value("base models with fine-tunes", "§5.5", sel.family.nunique(), 17)
checks.same("at most 100 selected per base model", "§5.5", bool(sel.groupby("family").size().max() <= 100), True)
checks.value("excluded: preliminary ROME edit did not succeed", "§5.5, App. C", int((~sel.rome_success).sum()), 43)
checks.value("excluded: could no longer be downloaded", "§5.5, App. C", int(fam.download_unavailable.sum()), 3)
checks.value("attempted (preliminary edit succeeded)", "§5.5", int(fam.attempted.sum()), 747)
checks.value("evaluated checkpoints", "§5.5, Abstract", len(ft), 744)
key = lambda d: set(zip(d.model_id, d.revision))
checks.same("evaluated checkpoints are among those whose preliminary edit succeeded", "§5.5",
            key(ft) <= key(sel[sel.rome_success]), True)
types = ft.checkpoint_type.value_counts()
checks.value("full checkpoints", "§5.5, App. C", int(types["full"]), 550)
checks.value("adapters", "§5.5, App. C", int(types["adapter"]), 194)
per_base = ft.groupby("family").size()
names = MODELS.set_index("key").name
checks.value("fewest checkpoints per base model", "§5.5", per_base.min(), 2)
checks.same("base model with the fewest", "§5.5", set(names[per_base[per_base == per_base.min()].index]), {"Ministral~3 8B"})
checks.value("most checkpoints per base model", "§5.5", per_base.max(), 85)
checks.same("base models with the most", "§5.5", set(names[per_base[per_base == per_base.max()].index]),
            {"Qwen3-4B", "Mistral-7B v0.3"})
checks.value("distinct CounterFact facts used", "§5.5 Protocol", ft.case_id.nunique(), 100)
checks.same("one edit per checkpoint", "§5.5 Protocol", bool(ft.groupby(["model_id", "revision"]).size().eq(1).all()), True)

## Table 4

Rows in the order of Table 3, followed by the total. Written to `outputs/table4_rows.tex`.

In [ ]:
per = ft.groupby("family").agg(n=("model_id", "size"), rome=("rome_success", "sum"),
                               tp=("edited_gram_exact", "sum"), fp=("baseline_fp", "sum"))
per = MODELS.set_index("key").join(per)
checks.same("per-checkpoint counts agree with owncov-family-results.csv", "consistency",
            per[["n", "rome", "tp", "fp"]].astype(int).values.tolist(),
            fam.loc[per.index, ["evaluated", "rome_success", "edited_gram_exact", "baseline_fp"]].astype(int).values.tolist())
checks.same("TP means the edited checkpoint returns l*", "definition",
            bool(((ft.edited_selected_layer == ft.target_layer) == ft.edited_gram_exact).all()), True)
checks.same("FP means the unedited checkpoint returns l*", "definition",
            bool(((ft.baseline_selected_layer == ft.target_layer) == ft.baseline_fp).all()), True)

def table4_rows(d):
    lines = []
    for i, (_, g) in enumerate(d.groupby("period", sort=False)):
        if i:
            lines.append(r"\addlinespace")
        for r in g.itertuples():
            lines.append(f"{r.name:<22} & {r.n:3d} & {r.rome:3d} & {r.tp:3d} & {r.fp:3d} \\\\")
    lines += [r"\midrule",
              f"{'Total':<22} & {d.n.sum():3d} & {d.rome.sum():3d} & {d.tp.sum():3d} & {d.fp.sum():3d} \\\\"]
    return "\n".join(lines)

PAPER_TABLE4 = r"""
GPT-2 Medium           &  63 &  63 &  50 &   0 \\
GPT-2 Large            &  48 &  46 &  47 &   0 \\
GPT-2 XL               &  21 &  21 &  21 &   0 \\
GPT-J-6B               &  10 &  10 &   9 &   0 \\
\addlinespace
OPT-6.7B               &   3 &   3 &   3 &   0 \\
Falcon-7B              &  36 &  36 &  30 &   0 \\
Llama-2-7B             &  18 &  18 &  18 &  17 \\
Mistral-7B v0.1        &  72 &  70 &  69 &  65 \\
DeepSeek-LLM-7B Base   &   8 &   8 &   8 &   0 \\
\addlinespace
Mistral-7B v0.3        &  85 &  82 &  80 &  77 \\
Qwen3-4B               &  85 &  84 &  82 &   0 \\
Qwen3-8B               &  82 &  80 &  81 &   0 \\
Granite~4.0 Micro      &  57 &  56 &   2 &   0 \\
Olmo~3 7B              &  70 &  70 &   1 &   0 \\
Ministral~3 8B         &   2 &   2 &   2 &   0 \\
\addlinespace
Qwen3.5-4B             &  80 &  80 &  78 &   0 \\
Granite~4.1 8B         &   4 &   4 &   3 &   0 \\
\midrule
Total                  & 744 & 733 & 584 & 159 \\
"""

rows4 = table4_rows(per)
(OUT / "table4_rows.tex").write_text(rows4 + "\n")
checks.same("Table 4 rows equal the paper", "Table 4", latex_rows_equal(rows4, PAPER_TABLE4), True)
print(rows4)
diff_rows(rows4, PAPER_TABLE4)

## Outcomes (§5.5, Abstract, Conclusion)

In [ ]:
n = len(ft)
checks.value("ROME successes", "§5.5", int(ft.rome_success.sum()), 733)
checks.printed("ROME success rate (%)", "§5.5", 100 * ft.rome_success.mean(), "98.5")
checks.value("edited checkpoints flagged (TP)", "§5.5", int(ft.edited_gram_exact.sum()), 584)
checks.printed("TP rate (%)", "§5.5, Conclusion; 79 percent in the Abstract", 100 * ft.edited_gram_exact.mean(), "78.5")
checks.value("unedited checkpoints flagged (FP)", "§5.5", int(ft.baseline_fp.sum()), 159)
checks.printed("FP rate (%)", "§5.5, Conclusion; 21 percent in the Abstract", 100 * ft.baseline_fp.mean(), "21.4")

FP_BASES = {"llama2-7b", "mistral-7b-v0.1", "mistral-7b-v0.3"}
checks.same("every FP is a fine-tune of Llama-2-7B or a Mistral-7B model", "§5.5",
            set(ft[ft.baseline_fp].family) <= FP_BASES, True)
other = ft[~ft.family.isin(FP_BASES)]
lm = ft[ft.family.isin(FP_BASES)]
checks.value("other 14 base models: FP", "§5.5", int(other.baseline_fp.sum()), 0)
checks.value("other 14 base models: TP", "§5.5", int(other.edited_gram_exact.sum()), 417)
checks.value("other 14 base models: checkpoints", "§5.5", len(other), 569)
checks.printed("other 14 base models: TP rate (%)", "§5.5", 100 * other.edited_gram_exact.mean(), "73.3")
checks.value("Llama-2/Mistral fine-tunes: TP", "§5.5", int(lm.edited_gram_exact.sum()), 167)
checks.value("Llama-2/Mistral fine-tunes: FP", "§5.5", int(lm.baseline_fp.sum()), 159)
checks.value("Llama-2/Mistral fine-tunes: checkpoints", "§5.5", len(lm), 175)

misses = ft[~ft.edited_gram_exact]
checks.value("misses", "§5.5", len(misses), 160)
checks.same("every miss returns the layer selected on the same unedited fine-tune", "§5.5",
            bool((misses.edited_selected_layer == misses.baseline_selected_layer).all()), True)

granite_micro = per.loc["granite4-micro"]; olmo = per.loc["olmo-3-1025-7b"]
checks.same("Granite 4.0 Micro: TP of n", "§5.5", (int(granite_micro.tp), int(granite_micro.n)), (2, 57))
checks.same("Olmo 3 7B: TP of n", "§5.5", (int(olmo.tp), int(olmo.n)), (1, 70))

## Comparison with the base models (§5.5)

The paper states that the per-model detection rate on fine-tunes stays within 10 percentage points of the exact rate on the base model (Table 3) for 15 of the 17 base models,
the exceptions being the two models with very few fine-tunes.

In [ ]:
base = base_model_results().set_index("key")
cmp = pd.DataFrame({"name": per.name, "n": per.n, "finetunes_tp_pct": 100 * per.tp / per.n, "base_exact_pct": base.exact_pct})
cmp["difference_pp"] = (cmp.finetunes_tp_pct - cmp.base_exact_pct).abs()
within = cmp.difference_pp <= 10
checks.value("base models within 10 percentage points", "§5.5", int(within.sum()), 15)
checks.same("exceptions", "§5.5", set(cmp[~within].name), {"OPT-6.7B", "Granite~4.1 8B"})
checks.same("their numbers of checkpoints", "§5.5", sorted(cmp[~within].n.astype(int)), [3, 4])
cmp

## Failed edits (§5.6)

In [ ]:
failed = ft[~ft.rome_success]
checks.value("fine-tune edits that did not reach ES = 1", "§5.6", len(failed), 11)
checks.value("of these, localized by the detector", "§5.6", int(failed.edited_gram_exact.sum()), 8)
failed[["family", "model_id", "rome_efficacy_score", "target_layer", "edited_selected_layer"]]

## Summary of checks

In [ ]:
checks.table().to_csv(OUT / "checks_04_finetunes.csv", index=False)
checks.assert_all()
checks.table()

In [ ]:
inputs_used()